# Technical Analysis agent (Juliet) – Team 12, QDC Fall 2026

**Question this agent answers:** *which ETFs have the price trend on their side right now, and is it a good moment to enter?*

**Design choices**
- **Pure Python, no AI call.** Same rule as the team graph: *the AI never computes a number*. It also means the signals still work when Gemini is busy or over its free quota.
- **Evidence first.** Each indicator was tested on our 50 ETFs (2022–2026) before being used (sections 4–5). Only the ones with a real, stable predictive power drive the view; the others are just timing notes.
- **Same data and rules as the team:** `data/prices.csv` (Morgane's clean file), today's intraday row never used (orders execute at the T-1 close), `adj_close` for indicators (no fake drop on ex-dividend days), `close` for returns in the backtest (dividends are not credited in the competition).

| Section | What |
|---|---|
| 1 | Setup |
| 2 | Today's signals per ETF |
| 3 | Technical view per theme (same format as the macro agent) |
| 4 | Evidence: which indicators predict the next 5–10 days? |
| 5 | Evidence: replaying the competition on history |
| 6 | How to plug the agent into `team_graph.ipynb` |
| 7 | Limits |

## 1. Setup
On Colab: upload `prices.csv` and `technical_agent.py` with the folder icon (or run from the repo, where both are found automatically).

In [1]:
import os, sys, shutil
from pathlib import Path
for folder in [Path.cwd(), *Path.cwd().parents]:           # project folder = the one with data/prices.csv
    if (folder / "data" / "prices.csv").exists():
        os.chdir(folder); break
Path("data").mkdir(exist_ok=True); Path("Technical_code").mkdir(exist_ok=True)
for f, dest in [("prices.csv", "data"), ("technical_agent.py", "Technical_code")]:
    if Path(f).exists() and not (Path(dest) / f).exists():  # Colab: move uploaded files into place
        shutil.move(f, Path(dest) / f)
sys.path.insert(0, "Technical_code")

import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
from technical_agent import (load_prices, run_technical_agent, ic_table, simulate_competitions,
                             simulation_summary, technical_conflicts, METHOD)
prices = load_prices("data/prices.csv")
print(f"{prices['code'].nunique()} ETFs, {prices['date'].min():%Y-%m-%d} to {prices['date'].max():%Y-%m-%d}")

50 ETFs, 2021-01-04 to 2026-11-13


## 2. Today's signals per ETF
- `tech_score`: average percentile rank (0 = worst, 1 = best) of **6-month** and **12-1-month** momentum across our ETFs (12-1 = the last 12 months without the last month, the classic momentum measure).
- `tech_view`: **positive** if `tech_score ≥ 0.67` **and** the price is above its 50-day average; **negative** if `≤ 0.33` **and** below; otherwise **neutral**. Requiring both avoids buying a long-term winner that is already falling (e.g. a top momentum ETF below its 50-day average stays neutral).
- `behaviour`: from each ETF's own history (10-day variance ratio, last 3 years): **trending** ETFs are timed with breakouts, **mean-reverting** ones with pullbacks.
- Leveraged/inverse ETFs (NASDAQ 3X, EURO STOXX 2X, S&P 500 SHORT, STOXX SHORT) are **not ranked**: they follow their unleveraged twin (inverse = flipped view), otherwise the same bet would be counted twice.

In [2]:
result = run_technical_agent(prices)
signals = result["signals"]
cols = ["theme", "tech_view", "confidence", "tech_score", "mom_rank", "mom_6m_%", "mom_12_1_%",
        "vs_ma50_%", "rsi14", "breakout_20d", "behaviour", "timing_note"]
print("As of", result["as_of"], "| views:", signals["tech_view"].value_counts().to_dict())
signals[cols]

As of 2026-11-13 | views: {'neutral': 32, 'negative': 10, 'positive': 8}


,theme,tech_view,confidence,tech_score,mom_rank,mom_6m_%,mom_12_1_%,vs_ma50_%,rsi14,breakout_20d,behaviour,timing_note
code,,,,,,,,,,,,
CYBERSEC,Equity / Tech,neutral,-,0.94,1.0,25.9,56.3,-3.4,43.0,False,trending,-
BIOTECH,Equity / Tech,positive,medium,0.90,2.0,28.8,30.7,3.8,54.0,False,mean-reverting,-
WORLD ENERGY,Energy & commodities,positive,high,0.87,3.0,11.2,51.8,1.3,50.0,False,trending,-
EURO BANKS,Financials,positive,high,0.83,4.0,14.7,29.5,0.4,58.0,False,mixed,-
IBEX 35,European equity,positive,medium,0.76,5.0,13.0,24.5,3.6,64.0,True,mean-reverting,-
ROBOTICS,Equity / Tech,neutral,-,0.76,5.0,4.0,42.5,-5.3,42.0,False,mixed,-
OIL & GAS E&P,Energy & commodities,positive,high,0.73,7.0,4.0,41.5,0.2,49.0,False,trending,-
DIVIDEND LEADERS,Size & style factors,positive,high,0.73,7.0,10.1,24.6,4.2,69.0,True,trending,breakout to a 20-day high (volume not confirmed)
HEALTH CARE,Health care,positive,high,0.71,9.0,28.1,16.8,9.7,71.0,False,mixed,overbought (RSI > 70): risk of buying after a sharp rise


## 3. Technical view per theme
Same format as Gonçalo's macro output (`theme → view, reason, sources`), so the Portfolio Manager and the validation can compare the two views theme by theme.

In [3]:
pd.DataFrame(result["theme_views"]).T[["view", "reason"]]

,view,reason
Defensive sector,neutral,"avg tech_score=0.51, 50% of its ETFs above their 50-day average (STAPLES, DE..."
Digital assets,negative,"avg tech_score=0.18, 0% of its ETFs above their 50-day average (BITCOIN, CRY..."
Emerging markets,neutral,"avg tech_score=0.46, 33% of its ETFs above their 50-day average (BRAZIL, KOR..."
Energy & commodities,positive,"avg tech_score=0.80, 100% of its ETFs above their 50-day average (WORLD ENER..."
Energy transition,negative,"avg tech_score=0.25, 0% of its ETFs above their 50-day average (HYDROGEN, CL..."
Equity / Tech,neutral,"avg tech_score=0.78, 20% of its ETFs above their 50-day average (CYBERSEC, B..."
European equity,neutral,"avg tech_score=0.65, 100% of its ETFs above their 50-day average (IBEX 35, E..."
Financials,positive,"avg tech_score=0.83, 100% of its ETFs above their 50-day average (EURO BANKS)"
Fixed income / Bonds,neutral,"avg tech_score=0.36, 100% of its ETFs above their 50-day average (HIGHY BOND..."
Frontier tech,negative,"avg tech_score=0.33, 0% of its ETFs above their 50-day average (SPACE, QUANTUM)"


## 4. Evidence: which indicators predict the next 5–10 trading days?
**Information coefficient (IC):** every 5 (or 10) days, rank our ETFs by an indicator and by their return over the next 5 (10) days, and measure the rank correlation. A useful indicator has a **mean IC > 0**, a **t-stat above ~2** (not luck) and is positive in most years.

Horizons of 5 and 10 days match the competition: 3–4 days between our trading days, 12 sessions in total. Future returns use **close prices** (what the competition pays).

In [4]:
ic = ic_table(prices)
Path("Technical_code/results").mkdir(parents=True, exist_ok=True)
ic.to_csv("Technical_code/results/ic_table.csv", index=False)
ic

,indicator,horizon_days,mean_IC,t_stat,hit_rate,n_periods,positive_years
5,tech_score,5,0.044,2.64,0.58,248,5/5
0,mom_12_1,5,0.049,2.49,0.58,248,5/5
1,mom_6m,5,0.035,1.98,0.55,248,5/5
4,rsi14,5,0.021,1.32,0.52,248,3/5
3,breakout_20d,5,0.016,1.25,0.52,229,3/5
2,vs_ma50,5,0.021,1.14,0.53,248,3/5
6,low_vol_60d,5,-0.027,-1.08,0.50,211,3/5
12,tech_score,10,0.052,2.05,0.57,124,5/5
7,mom_12_1,10,0.047,1.71,0.53,124,5/5
11,rsi14,10,0.029,1.49,0.56,124,4/5


**Reading:** the combined momentum score (`tech_score`) is the best and most stable indicator (positive in all 5 years, t-stat > 2 at both horizons). Trend (`vs_ma50`), breakout and RSI have a small positive IC but are **not significant**; low volatility is even slightly **negative** in our universe. That is why momentum drives the view and the others are only used as timing notes.

Note: an IC of ~0.05 is small but normal for a real signal in finance: it is a probability edge, not a forecast.

## 5. Evidence: replaying the competition on history
We replay **our competition** every week since 2022 (243 windows), with its real rules: 5 trading days in 12 sessions (same pattern as 7–21 Dec), max 5 orders per day, orders at the previous close, valuation at close prices (no dividends), no leverage.

Pure-technical test strategy: equal weight in the 5 ETFs with the best `tech_score` that are above their 50-day average; on later trading days, replace at most 2 that dropped out. Benchmark: 100% WORLD.

In [5]:
sim = simulate_competitions(prices)
sim.to_csv("Technical_code/results/competition_replay.csv", index=False)
print(simulation_summary(sim))
sim.assign(year=pd.to_datetime(sim["start"]).dt.year).groupby("year")[["strategy_%", "world_%", "excess_%"]].mean().round(2)

{'windows': 243, 'beat_WORLD_%': 49.4, 'avg_excess_%': 0.6, 'median_excess_%': -0.07, 'avg_strategy_%': 1.22, 'avg_world_%': 0.61, 'worst_strategy_%': -12.32, 'worst_world_%': -12.9, 'best_strategy_%': 19.52}


,strategy_%,world_%,excess_%
year,,,
2022,0.37,-0.31,0.68
2023,-0.07,0.85,-0.92
2024,1.57,1.25,0.32
2025,2.93,0.37,2.56
2026,1.25,0.85,0.40


**Reading (honest):** on average the technical strategy earns more than WORLD per competition window, but it beats WORLD only about **half of the time** (median excess ≈ 0): the gain comes from a few strong trending periods (2025), and it lost in 2023. Alone, momentum is **not** enough to beat the benchmark reliably over 2–3 weeks.

**Consequence for our system:** the technical signal is one input for the Portfolio Manager, next to the macro view and the risk rules, not an autopilot. It is most useful to (1) avoid ETFs in a clear downtrend (negative views) and (2) choose between ETFs inside a theme the team already likes.

## 6. How to plug it into `team_graph.ipynb` (Francesco)
Francesco's `build_pm_prompt(pf, macro, technical_signals=...)` already has a slot for this agent. Three small changes:

In [6]:
import json
# --- 1) a node, after macro (uses the same in-memory `prices` as the other agents) ---------------------------
#   State: add the key   technical: dict
def technical_node(state):
    print("1b) Technical: momentum, trend, breakout, RSI (Python, no AI)...", flush=True)
    r = run_technical_agent(prices)
    return {"technical": {"as_of": r["as_of"], "summary": r["summary"], "theme_views": r["theme_views"],
                          "signals": json.loads(r["signals"].reset_index().to_json(orient="records", date_format="iso"))}}

#   graph:  g.add_node("technical", technical_node)
#           g.add_edge("macro", "technical");  g.add_edge("technical", "portfolio_manager")   (instead of macro -> portfolio_manager)

# --- 2) in portfolio_manager_node: pass the summary to the prompt ----------------------------------------------
#   r = ask_portfolio_manager(state["portfolio"], state["macro"],
#                             technical_signals=state["technical"]["summary"], feedback=feedback)

# --- 3) in validate_node: same idea as the macro check ---------------------------------------------------------
#   buying an ETF with a NEGATIVE technical view needs "Contrary to technical signal because ..." in its reason
#   sig = pd.DataFrame(state["technical"]["signals"]).set_index("code")
#   errors += technical_conflicts(state["weights"], sig, state["reasons"], current_weights(state["portfolio"]))

# quick test of the check (no AI): buying RARE EARTHS-like negative ETFs without a reason is flagged
neg = signals.index[signals["tech_view"] == "negative"][:1].tolist()
print("Test - buy a negative-view ETF without explanation:", technical_conflicts({neg[0]: 0.10}, signals, {}))
print("Test - same, with 'Contrary to technical signal because ...':",
      technical_conflicts({neg[0]: 0.10}, signals, {neg[0]: "Contrary to technical signal because the macro view is strong"}))
print("\nWhat the Portfolio Manager receives (first lines):\n" + result["summary"][:1200] + " ...")

Test - buy a negative-view ETF without explanation: ["GOLD: its technical view is NEGATIVE (tech_score=0.31 (momentum rank 34 of 45), mom_6m=-9.6%, mom_12_1=5.6%, vs_ma50=-4.3%, rsi14=41) - remove it or explain in its reason: 'Contrary to technical signal because ...'"]
Test - same, with 'Contrary to technical signal because ...': []

What the Portfolio Manager receives (first lines):
Computed in Python by the Technical Analysis agent from adjusted closing prices (T-1 close). tech_score = average percentile rank (0 worst, 1 best) of 6-month and 12-1-month momentum across our ETFs: in our 2022-2026 backtest this was the ONLY indicator with a stable predictive power for the next 5-10 trading days. tech_view: positive = tech_score >= 0.67 AND price above its 50-day average; negative = tech_score <= 0.33 AND below; else neutral. Leveraged/inverse ETFs follow their unleveraged twin (inverse = flipped). breakout_20d, rsi14 and timing_note are weak signals on their own: use them only to time 

## 7. Limits (for the report)
- **Momentum fails at turning points** (e.g. sharp reversals after news): the macro view and the human approval are there for that.
- **Short histories** (QUANTUM, BITCOIN, NASDAQ INCOME…) have a 6-month signal only; ETFs with < 127 days of data get no view.
- **Leveraged ETFs** reset daily: their long-run return is not 3× the index when markets go sideways, so they never get "high" confidence.
- The behaviour type (trending / mean-reverting) is estimated from 3 years of data and can change.
- **Multiple testing:** we tested 7 indicators × 2 horizons; we only rely on the one that is significant at both horizons and positive every year, to reduce the risk of a lucky result.